# Face verification & triplet loss

_Deep Learning_

**Learn an encoding where the same person's faces sit close and different people sit far.**

Face recognition turns a face photo into a list of numbers called an encoding.

     Two photos of the same person should get encodings that are close together. Two different people should be far apart.

---

This notebook is a practice scaffold for the **Face verification & triplet loss** lesson. We build it up one small step at a time: run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Make an anchor, a positive, and a negative

The triplet loss works on three embeddings at once. The **anchor** is one photo of person A; the **positive** is another photo of the same person A (so we build it as the anchor plus a little noise); and the **negative** is a photo of a different person B (an independent random vector). We `F.normalize` the anchor and negative so they sit on the unit sphere, just like real face encodings.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# pretend an encoder produced these 128-d face embeddings
anchor = F.normalize(torch.randn(1, 128), dim=1)    # a photo of person A
positive = anchor + 0.05 * torch.randn(1, 128)      # another photo of person A
negative = F.normalize(torch.randn(1, 128), dim=1)  # a photo of person B

### Step 2 — Measure the two distances

Before computing the loss, let's look at the raw distances it is built from. The anchor-to-positive distance should be **small** (same person), and the anchor-to-negative distance should be **large** (different people). These are the two quantities the triplet loss compares.

In [ ]:
d_pos = (anchor - positive).norm().item()    # should be small
d_neg = (anchor - negative).norm().item()    # should be large
print("dist(anchor, positive):", round(d_pos, 3))
print("dist(anchor, negative):", round(d_neg, 3))

### Step 3 — Compute the triplet loss

`nn.TripletMarginLoss(margin=0.2)` returns `max(0, d_pos - d_neg + margin)`. It is zero once the negative is at least a `margin` farther than the positive — meaning the encoding already separates the two identities well — and positive otherwise, pushing the negative away and pulling the positive closer.

In [ ]:
triplet = nn.TripletMarginLoss(margin=0.2)   # pull anchor-positive close, push negative away
loss = triplet(anchor, positive, negative)
print("triplet loss:", round(loss.item(), 4))

## Visualize the data & results

_Does a learned encoding pull images of the same digit-class together and push different classes apart?_

### Step 1 — Pick two 'identities' from the digit images

Real face data is hard to ship, so we borrow the digit images as stand-in identities: all the 3s are 'person 3' and all the 8s are 'person 8'. We scale the pixel values to 0–1 and keep only the rows whose label is 3 or 8.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

digits = load_digits()
X = digits.data / 16.0
y = digits.target

mask = np.isin(y, [3, 8])                # two 'identities': 3s and 8s

### Step 2 — Embed the images down to 2-D

Each image has 64 pixels, far too many to plot. We use PCA to compress the masked images down to two components — a simple stand-in for the learned encoding a real face network would produce — and keep the matching labels so we can colour each point by identity.

In [ ]:
Z = PCA(n_components=2, random_state=1).fit_transform(X[mask])
ym = y[mask]

### Step 3 — Plot the two identity clusters

We scatter the 2-D points, colouring the 3s and 8s separately. If the encoding is doing its job, the two identities form distinct clusters — same-identity points sit close together and different identities sit apart, exactly the behaviour the triplet loss rewards.

In [ ]:
plt.scatter(Z[ym == 3][:, 0], Z[ym == 3][:, 1], color="#4ea1ff", label="identity: 3")
plt.scatter(Z[ym == 8][:, 0], Z[ym == 8][:, 1], color="#7ee787", label="identity: 8")
plt.title("Real digit images (3s vs 8s) embedded to 2-D, two identities cluster apart")
plt.xlabel("component 1")
plt.ylabel("component 2")
plt.legend()
plt.show()